# Eksperimen 2: Baseline Embedding (Frozen SBERT)

**GEMASTIK KTI 2026** | Tim Peneliti

Eksperimen ini menguji apakah model bahasa pralatih (pretrained) yang menangkap representasi semantik dasar sudah cukup memadai untuk melakukan penilaian, tanpa perlu penyesuaian khusus (fine-tuning). Model SBERT multibahasa digunakan untuk mengekstrak vektor semantik berdimensi 384 dari jawaban siswa dan kunci referensi. Serupa dengan Eksperimen 1, tiga metrik jarak (Cosine, Euclidean, dan Continuous Jaccard) dikalkulasi dari vektor embedding tersebut, kemudian diintegrasikan ke dalam algoritma SVR.

## 0. Persiapan Lingkungan dan Konfigurasi

In [ ]:
import sys
import os

try:
    import google.colab
    IN_COLAB = True
    print("Lingkungan Eksekusi: Google Colab")
    if not os.path.exists("/content/indo-asag"):
        os.system("git clone https://github.com/Rnov24/indo-asag.git /content/indo-asag")
    os.system("pip install -q -e /content/indo-asag[all]")
    REPO_ROOT = "/content/indo-asag"
except ImportError:
    IN_COLAB = False
    try:
        REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
    except NameError:
        REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    print(f"Lingkungan Eksekusi: Lokal ({REPO_ROOT})")

sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from indo_asag.data import load_dataset
from indo_asag.evaluation import run_multi_seed
from indo_asag.utils import set_seed, load_config
from indo_asag.utils.github import auto_push_to_github

config = load_config(os.path.join(REPO_ROOT, "configs", "base.yaml"))
SEEDS = config["seeds"]
RESULTS_DIR = os.path.join(REPO_ROOT, "results", "prelim")
PREDS_DIR = os.path.join(RESULTS_DIR, "predictions")

## 1. Pemuatan Dataset

In [ ]:
DATA_PATH = os.path.join(REPO_ROOT, config["data"]["path"])
df = load_dataset(DATA_PATH)

## 2. Eksekusi Eksperimen 2

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

print("\n" + "=" * 60)
print("EXP 2: Frozen SBERT (Cosine, Euclidean, Jaccard + SVR)")
print("=" * 60)

sbert_model = SentenceTransformer(config["features"]["sbert_model"])

print("Mengubah jawaban menjadi embedding (encoding)...")
ans_emb = sbert_model.encode(df["student_answer"].tolist(), batch_size=64,
                              normalize_embeddings=True, show_progress_bar=True)
ref_emb = sbert_model.encode(df["reference_answer"].tolist(), batch_size=64,
                              normalize_embeddings=True, show_progress_bar=True)

# Compute distances
cos_sim = np.sum(ans_emb * ref_emb, axis=1)
euc_dist = np.linalg.norm(ans_emb - ref_emb, axis=1)
tanimoto_jac = cos_sim / (2.0 - cos_sim)

df["sbert_cos"] = cos_sim
df["sbert_euc"] = euc_dist
df["sbert_jac"] = tanimoto_jac

def exp2_predict(train_df, val_df, fold, seed=42):
    features = ["sbert_cos", "sbert_euc", "sbert_jac"]
    X_tr = train_df[features].values
    X_va = val_df[features].values
    
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s = sc.transform(X_va)
    
    svr = SVR(
        kernel=config["model"]["svr"]["kernel"],
        C=config["model"]["svr"]["C"],
        gamma=config["model"]["svr"]["gamma"],
        epsilon=config["model"]["svr"]["epsilon"]
    )
    svr.fit(X_tr_s, train_df["score_norm"].values)
    return svr.predict(X_va_s)

exp2_summary = run_multi_seed(df, exp2_predict, seeds=SEEDS)
print(f"  QWK: {exp2_summary['QWK']}")

## 3. Penyimpanan Prediksi (Out-of-Fold)

In [ ]:
print("\nMenyimpan array prediksi OOF ke disk...")
for s, preds in exp2_summary["_preds"].items():
    np.save(os.path.join(PREDS_DIR, f"exp2_sbert_oof_seed{s}.npy"), preds)
print("[OK] Prediksi berhasil disimpan.")

## 4. Publikasi Otomatis ke GitHub

In [ ]:
auto_push_to_github("chore(auto): menyimpan prediksi Eksperimen 2 SBERT", IN_COLAB, REPO_ROOT)